# M1 Notebook 23 — Convex Optimization

**Status:** Runnable first edition

## Learning objectives

- Recognize convex sets and functions.
- Use projected gradient descent.
- Interpret duality and KKT conditions.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_math.optimization import (
    is_convex_quadratic,kkt_residual,project_simplex,projected_gradient_descent,
    quadratic_gradient,quadratic_value,
)


## Convex sets and functions

A set \(C\) is convex if every line segment between two points in \(C\) remains in \(C\).

A function is convex if

\[
f(\theta x+(1-\theta)y)\le \theta f(x)+(1-\theta)f(y).
\]


In [ ]:
Q=np.array([[3.,.5],[.5,1.]])
assert is_convex_quadratic(Q)
f=lambda x: quadratic_value(x,Q,np.array([-1.,-1.]))
g=lambda x: quadratic_gradient(x,Q,np.array([-1.,-1.]))


## Projected gradient descent on the simplex

In [ ]:
projection=lambda x: project_simplex(x,1.0)
x,h,it=projected_gradient_descent(f,g,[.5,.5],projection,learning_rate=.2,max_iter=500)
{"solution":x,"sum":x.sum(),"objective":h[-1],"iterations":it}


In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
ax.semilogy(h)
ax.set_xlabel("Iteration"); ax.set_ylabel("Objective")
ax.set_title("Projected Gradient Convergence")
plt.show()


## Duality intuition

Dual variables can be interpreted as shadow prices: the marginal value of relaxing constraints.

In [ ]:
budget=np.linspace(.5,2.0,100)
optimal=[]
for B in budget:
    best=np.inf
    for x1 in np.linspace(0,B,300):
        x2=B-x1
        val=f([x1,x2])
        best=min(best,val)
    optimal.append(best)
shadow=-np.gradient(optimal,budget)
pd.DataFrame({"budget":budget[::20],"optimal_value":np.array(optimal)[::20],"shadow_price":shadow[::20]})


## KKT residual check

In [ ]:
A=np.array([[1.,1.],[-1.,0.],[0.,-1.]])
b=np.array([1.,0.,0.])
lam=np.array([.2,0.,0.])
kkt_residual(g(x),A,x,b,lam)


## Decision Intelligence case

Choose a convex mix of intervention portfolios under a fixed total allocation.

In [ ]:
returns=np.array([1.0,1.3,.9])
risk=np.array([[1,.2,.1],[.2,1.5,.3],[.1,.3,.8]])
obj=lambda w: float(.5*w@risk@w-.8*returns@w)
grad=lambda w: risk@w-.8*returns
w,h,_=projected_gradient_descent(obj,grad,[1/3]*3,lambda z: project_simplex(z,1),learning_rate=.2,max_iter=500)
pd.Series(w,index=["Agriculture","Health","Energy"],name="allocation")


## Key insight

Convexity turns local information into global guarantees and makes constrained optimization substantially more reliable.